# Experiment 09 — CatBoost as a third blend parent (last attempt)

Every gain that *transferred* to the leaderboard in this project came from combining learners with **different errors** (the per-family blend in 04, the file-mix with the recency-rich model). This is the final score experiment: add **CatBoost** — a genuinely different gradient-boosting library (native categorical handling, ordered boosting, symmetric trees) — as a new, diverse parent, and blend it with our deployed best.

CatBoost is trained **exactly like our recursive LightGBM `geo` model**: same feature set (BASE + GEO), same day-by-day iterative forecasting (lags recomputed from own predictions), same 3 honest windows. The only changed variable is the learner. We then sweep the blend weight of `cat` against `geo_blend25`.

**Submission rule (agreed up front, given this project's hard lesson):** a candidate is submitted **only if it beats `geo_blend25` on all 3 windows by a *structural* margin (≥0.003 mean)** — not a sub-noise win. We have shown three times that validation gains below ~0.002–0.003 do not transfer to the public test. If CatBoost does not clear that bar, we lock in the current best (public **0.38803**) and stop.

> 🇷🇺 Последняя попытка по скору. Всё, что у нас *переносилось* на лидерборд, шло от бленда learner'ов с разными ошибками. Добавляем **CatBoost** — принципиально другой бустинг (нативные категории, ordered boosting, симметричные деревья) — как новый разнообразящий родитель. Обучаем точно как нашу рекурсивную LightGBM `geo` (тот же набор фич BASE+GEO, тот же итеративный прогноз, те же 3 окна); меняется только learner. Свип веса бленда `cat` против `geo_blend25`. **Правило сабмита:** только если бьёт `geo_blend25` на всех 3 окнах с *структурным* запасом (≥0.003 по среднему) — не под-шумовой выигрыш. Иначе фиксируемся на 0.38803.

In [1]:
import time
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, joblib
from catboost import CatBoostRegressor

In [2]:
_t0 = time.time()

MODELS_DIR = "../models/exp09"; ART_DIR = "../artifacts/exp09"; EXP05_ART = "../artifacts/exp05"
os.makedirs(MODELS_DIR, exist_ok=True); os.makedirs(ART_DIR, exist_ok=True)

CAT_PARAMS = dict(iterations=2500, learning_rate=0.04, depth=8, l2_leaf_reg=3.0,
                  loss_function="RMSE", random_seed=42, thread_count=-1, verbose=0)

BASE = [
    "day_of_week", "month", "year", "is_weekend", "day_of_year", "week_of_year", "date_index",
    "sin_day", "cos_day", "sin_week", "cos_week",
    "lag_1", "lag_2", "lag_3", "lag_4", "lag_5", "lag_6",
    "lag_7", "lag_14", "lag_21", "lag_28", "lag_42", "lag_56", "lag_364", "lag_365",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28", "rolling_mean_364",
    "dcoilwtico", "dcoilwtico_ma7", "dcoilwtico_ma28",
    "onpromotion", "onpromotion_ma7", "onpromotion_ma28",
    *[f"transactions_lag_{l}" for l in range(16, 24)],
    "is_holiday_national", "day_before_holiday", "is_black_friday", "is_cyber_monday", "is_terremoto",
    "is_navidad", "is_dia_madre", "is_futbol", "is_dia_trabajo", "is_primer_dia", "is_dia_difuntos", "work_day",
    "store_nbr", "store_type", "cluster", "family",
]
GEO = ["city", "state", "is_holiday_local", "is_holiday_regional"]
FEATURES = BASE + GEO
CAT = ["store_nbr", "store_type", "cluster", "family", "day_of_week", "month", "city", "state"]

WINDOWS = [("W1", pd.Timestamp("2017-06-15"), pd.Timestamp("2017-06-30")),
           ("W2", pd.Timestamp("2017-07-15"), pd.Timestamp("2017-07-30")),
           ("W3", pd.Timestamp("2017-07-31"), pd.Timestamp("2017-08-15"))]
WNAMES = [w for w, _, _ in WINDOWS]
TEST_START, TEST_END = pd.Timestamp("2017-08-16"), pd.Timestamp("2017-08-31")
SUBMIT_MARGIN = 0.003   # structural-gain bar for submitting / порог структурного выигрыша для сабмита
print(f"{len(FEATURES)} features; CatBoost {CAT_PARAMS['iterations']} iters depth {CAT_PARAMS['depth']}")

63 features; CatBoost 2500 iters depth 8


In [3]:
# Load parquet, reshape into (dates x series) matrices (same recursive engine as exp 04/05)
# Загружаем parquet, матрицы (даты x ряды) — тот же рекурсивный движок, что в 04/05
DF = (pd.read_parquet("../artifacts/features.parquet")
      .sort_values(["date", "store_nbr", "family"]).reset_index(drop=True))
DATES = np.sort(DF["date"].unique()); N_DATES = len(DATES); N_SERIES = DF.groupby("date").size().iloc[0]
assert len(DF) == N_DATES * N_SERIES
DT = pd.to_datetime(DATES); DATE_IDX = {d: i for i, d in enumerate(DATES)}
ref = DF.iloc[:N_SERIES][["store_nbr", "family"]].reset_index(drop=True)
SALES_TRUE = DF["sales"].to_numpy(float).reshape(N_DATES, N_SERIES)
DATE_ARR = DF["date"].to_numpy()
# CatBoost wants categorical columns as strings with no NaN; cast once on a working copy
# CatBoost хочет категориальные как строки без NaN; приводим один раз на рабочей копии
DFC = DF.copy()
for c in CAT:
    DFC[c] = DFC[c].astype(str)
print(f"{N_DATES} dates x {N_SERIES} series")

1704 dates x 1782 series


In [4]:
# Recursive engine (lags/rollings recomputed from own predictions), CatBoost learner
# Рекурсивный движок (лаги/rolling из собственных предсказаний), learner CatBoost
LAGS = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]; ROLLS = [7, 14, 28]


def rmsle_mat(P, Y):
    return float(np.sqrt(np.mean((np.log1p(np.clip(P, 0, None)) - np.log1p(Y)) ** 2)))


def dynamic_features(sm, i):
    feat = {f"lag_{k}": sm[i - k] for k in LAGS}
    for w in ROLLS:
        feat[f"rolling_mean_{w}"] = np.nanmean(sm[i - w:i], axis=0)
    win = sm[i - 364:i]; cnt = (~np.isnan(win)).sum(0)
    rm = np.full(N_SERIES, np.nan); ok = cnt >= 30
    if ok.any():
        rm[ok] = np.nanmean(win[:, ok], axis=0)
    feat["rolling_mean_364"] = rm
    return feat


def x_for_day(i, sm):
    X = DFC.iloc[i * N_SERIES:(i + 1) * N_SERIES][FEATURES].copy()
    for c, v in dynamic_features(sm, i).items():
        X[c] = v
    return X


def zero_rule_mask(first_idx):
    return np.nansum(SALES_TRUE[first_idx - 21:first_idx], axis=0) == 0


def run_iterative(predict_day, win_idxs, zmask):
    sm = SALES_TRUE.copy(); sm[win_idxs, :] = np.nan
    P = np.empty((len(win_idxs), N_SERIES))
    for j, i in enumerate(win_idxs):
        p = np.clip(predict_day(x_for_day(i, sm)), 0, None)
        p[zmask] = 0.0; sm[i] = p; P[j] = p
    return P


def train_cat(cutoff, tag):
    path = os.path.join(MODELS_DIR, f"cat_{tag}.cbm")
    if os.path.exists(path):
        m = CatBoostRegressor(); m.load_model(path); return m
    mask = (DATE_ARR >= np.datetime64("2016-01-01")) & (DATE_ARR < np.datetime64(cutoff)) & DF["sales"].notna().to_numpy()
    X = DFC.loc[mask, FEATURES]; y = np.log1p(DF.loc[mask, "sales"])
    t = time.time(); m = CatBoostRegressor(**CAT_PARAMS); m.fit(X, y, cat_features=CAT)
    m.save_model(path); print(f"  cat_{tag}: {mask.sum():,} rows, {time.time()-t:.0f}s"); return m


def pred_cat(m):
    return lambda X: np.expm1(m.predict(X[FEATURES]))


def logblend(A, B, w):
    return np.expm1(w * np.log1p(np.clip(A, 0, None)) + (1 - w) * np.log1p(np.clip(B, 0, None)))

print("engine ready")

engine ready


In [5]:
# Main runs: CatBoost recursive per window; compare to geo (LGBM) and geo_blend25
# Основные прогоны: CatBoost рекурсивно по окнам; сравнение с geo (LGBM) и geo_blend25
RESULTS, rows, WIN_Y = {}, [], {}
for wname, ws, we in WINDOWS:
    print(f"\n=== {wname}: {ws.date()} .. {we.date()} ===")
    win_idxs = [i for i in range(N_DATES) if ws <= DT[i] <= we]
    assert len(win_idxs) == 16
    Y = SALES_TRUE[win_idxs]; zmask = zero_rule_mask(win_idxs[0]); WIN_Y[wname] = Y
    RESULTS[(wname, "geo")] = np.load(os.path.join(EXP05_ART, f"pred_{wname}__geo.npy"))
    RESULTS[(wname, "geo_blend25")] = np.load(os.path.join(EXP05_ART, f"pred_{wname}__geo_blend25.npy"))

    m = train_cat(ws, wname)
    cache = os.path.join(ART_DIR, f"pred_{wname}__cat.npy")
    if os.path.exists(cache):
        P_cat = np.load(cache)
    else:
        t = time.time(); P_cat = run_iterative(pred_cat(m), win_idxs, zmask)
        np.save(cache, P_cat); print(f"  cat iterative: {time.time()-t:.0f}s")
    RESULTS[(wname, "cat")] = P_cat
    for sch in ["geo", "geo_blend25", "cat"]:
        sc = rmsle_mat(RESULTS[(wname, sch)], Y)
        rows.append({"window": wname, "scheme": sch, "rmsle": sc})
        print(f"  {sch:12s} RMSLE={sc:.5f}")

res = pd.DataFrame(rows); res.to_csv(os.path.join(ART_DIR, "results.csv"), index=False)
print(f"\nElapsed: {(time.time()-_t0)/60:.1f} min")


=== W1: 2017-06-15 .. 2017-06-30 ===
  geo          RMSLE=0.38272
  geo_blend25  RMSLE=0.38173
  cat          RMSLE=0.38527

=== W2: 2017-07-15 .. 2017-07-30 ===
  geo          RMSLE=0.38783
  geo_blend25  RMSLE=0.38652
  cat          RMSLE=0.39173

=== W3: 2017-07-31 .. 2017-08-15 ===
  geo          RMSLE=0.40369
  geo_blend25  RMSLE=0.40187
  cat          RMSLE=0.41586

Elapsed: 0.1 min


In [6]:
# Blend-weight sweep: cat blended with geo_blend25
# Свип веса: cat в бленде с geo_blend25
gb = {wn: rmsle_mat(RESULTS[(wn, "geo_blend25")], WIN_Y[wn]) for wn in WNAMES}
print(f"CatBoost solo vs geo (LGBM):  " + "  ".join(
    f"{wn}: {rmsle_mat(RESULTS[(wn,'cat')], WIN_Y[wn]):.5f} vs {rmsle_mat(RESULTS[(wn,'geo')], WIN_Y[wn]):.5f}" for wn in WNAMES))
print(f"\n{'w_cat':>6} | " + "  ".join(f"{w:>9}" for w in WNAMES) + f" | {'mean':>8} | gate")
sweep = {}
for w in [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    sc = {wn: rmsle_mat(logblend(RESULTS[(wn, "cat")], RESULTS[(wn, "geo_blend25")], w), WIN_Y[wn]) for wn in WNAMES}
    sweep[w] = sc; d = {wn: sc[wn] - gb[wn] for wn in WNAMES}
    g = "PASS" if all(d[wn] < 0 for wn in WNAMES) else "fail"
    print(f"{w:>6.2f} | " + "  ".join(f"{d[wn]:+8.5f}" for wn in WNAMES) + f" | {np.mean(list(sc.values())):.5f} | {g}")

PASSW = [w for w in sorted(sweep) if all(sweep[w][wn] < gb[wn] for wn in WNAMES)]
BLEND_W = min(PASSW, key=lambda w: np.mean(list(sweep[w].values()))) if PASSW else None
gb_mean = np.mean(list(gb.values()))
margin = (gb_mean - np.mean(list(sweep[BLEND_W].values()))) if BLEND_W else 0.0
STRUCTURAL = BLEND_W is not None and margin >= SUBMIT_MARGIN
print(f"\nGate-passing weights: {PASSW}")
print(f"Best weight: {BLEND_W}, mean-margin vs geo_blend25: {margin:+.5f} "
      f"(submit bar {SUBMIT_MARGIN}) -> {'SUBMIT' if STRUCTURAL else 'DO NOT SUBMIT (sub-noise / no pass)'}")

CatBoost solo vs geo (LGBM):  W1: 0.38527 vs 0.38272  W2: 0.39173 vs 0.38783  W3: 0.41586 vs 0.40369

 w_cat |        W1         W2         W3 |     mean | gate
  0.20 | -0.00088  -0.00030  +0.00074 | 0.38990 | fail
  0.25 | -0.00098  -0.00027  +0.00109 | 0.38999 | fail
  0.30 | -0.00103  -0.00019  +0.00151 | 0.39014 | fail
  0.35 | -0.00102  -0.00008  +0.00199 | 0.39034 | fail
  0.40 | -0.00097  +0.00008  +0.00253 | 0.39059 | fail
  0.50 | -0.00071  +0.00052  +0.00381 | 0.39125 | fail

Gate-passing weights: []
Best weight: None, mean-margin vs geo_blend25: +0.00000 (submit bar 0.003) -> DO NOT SUBMIT (sub-noise / no pass)


In [7]:
# Final: build submission only if the gain is structural (>= 0.003 on the gate)
# Финал: собрать сабмишн только если выигрыш структурный (>= 0.003 на гейте)
if not STRUCTURAL:
    print(f"CatBoost blend does not clear the {SUBMIT_MARGIN} structural bar — no submission.")
    print("Per the agreed rule, we lock in the current best (public 0.38803).")
else:
    test_idxs = [i for i in range(N_DATES) if TEST_START <= DT[i] <= TEST_END]
    zt = zero_rule_mask(test_idxs[0])
    m_full = train_cat(TEST_START, "FULL")
    P_cat_test = run_iterative(pred_cat(m_full), test_idxs, zt)
    np.save(os.path.join(ART_DIR, "pred_TEST__cat.npy"), P_cat_test)
    test = pd.read_csv("../data/test.csv", parse_dates=["date"])
    long = pd.concat([pd.DataFrame({"date": DATES[i], "store_nbr": ref["store_nbr"].values,
                                    "family": ref["family"].values, "sales": P_cat_test[j]})
                      for j, i in enumerate(test_idxs)], ignore_index=True)
    cat_sub = test.merge(long, on=["date", "store_nbr", "family"], how="left")[["id", "sales"]].sort_values("id").reset_index(drop=True)
    geo = pd.read_csv("../submission_05_geo_blend25.csv").sort_values("id").reset_index(drop=True)
    out = geo[["id"]].copy(); out["sales"] = logblend(cat_sub["sales"].to_numpy(), geo["sales"].to_numpy(), BLEND_W)
    out.to_csv("../submission_09_cat_blend.csv", index=False)
    print(f"Saved submission_09_cat_blend.csv (w_cat={BLEND_W}, total {out['sales'].sum():,.0f})")
print(f"\nTotal runtime: {(time.time()-_t0)/60:.1f} min")

CatBoost blend does not clear the 0.003 structural bar — no submission.
Per the agreed rule, we lock in the current best (public 0.38803).

Total runtime: 0.1 min


## Conclusions

**Negative result — CatBoost does not help, and the gate rejected it automatically. No submission. This was the agreed last score experiment; the project's best stays at public 0.38803.**

| Scheme | W1 | W2 | W3 (Aug) | mean |
|---|---:|---:|---:|---:|
| geo_blend25 (baseline) | 0.38173 | 0.38652 | **0.40187** | 0.39004 |
| geo (LightGBM solo) | 0.38272 | 0.38783 | 0.40369 | 0.39141 |
| cat (CatBoost solo) | 0.38527 | 0.39173 | **0.41586** | 0.39762 |

CatBoost, trained exactly like our LightGBM `geo` model (same features, same recursive engine, only the learner changed), is weaker on every window and much weaker on W3 — the August window that mirrors the public test (0.41586 vs 0.40369). The weight sweep is clear: a small CatBoost dose lowers W1/W2 slightly but raises W3 at every weight, so no weight beats `geo_blend25` on all three windows.

**Why it failed — the wrong kind of diversity.** A blend helps only when a model's errors are different and not systematically worse in the target region. CatBoost is both weaker overall and worse exactly where it counts (August). A second library is not automatically a useful ensemble member.

**End of the score search.** Across notebooks 06–09, four ideas in a row (linear direct, dynamic features, direct-GBM, CatBoost) failed to beat the deployed best on a transferable basis. **The practical ceiling of this honest single-pipeline approach is ≈0.388, and the public best of 0.38803 is locked in.** The gap to the ~0.3735 "wall" is a heavily-forked tuned ensemble, not a target for single-model work. The lasting value of this project is the methodology: leak-free iterative validation, the 3-window acceptance gate, and a documented map of what transfers to the leaderboard and what does not.

## Выводы

**Отрицательный результат — CatBoost не помогает, и гейт отклонил его автоматически. Сабмишна нет. Это была согласованная последняя попытка улучшить скор; лучший результат проекта остаётся на паблике 0.38803.**

| Схема | W1 | W2 | W3 (авг) | среднее |
|---|---:|---:|---:|---:|
| geo_blend25 (бейзлайн) | 0.38173 | 0.38652 | **0.40187** | 0.39004 |
| geo (LightGBM соло) | 0.38272 | 0.38783 | 0.40369 | 0.39141 |
| cat (CatBoost соло) | 0.38527 | 0.39173 | **0.41586** | 0.39762 |

CatBoost, обученный точно как наша LightGBM `geo` (те же признаки, тот же рекурсивный движок, сменён только learner), слабее на каждом окне и сильно слабее на W3 — августовском окне, которое отражает публичный тест (0.41586 против 0.40369). Свип веса однозначен: малая доза CatBoost чуть снижает W1/W2, но повышает W3 при любом весе, поэтому ни один вес не бьёт `geo_blend25` на всех трёх окнах.

**Почему не получилось — не то разнообразие.** Бленд помогает только когда ошибки модели другие и не систематически хуже в целевой области. CatBoost и слабее в целом, и хуже именно там, где важно (август). Вторая библиотека не становится полезным членом ансамбля автоматически.

**Конец гонки за скором.** В ноутбуках 06–09 четыре идеи подряд (линейная direct, динамические фичи, direct-GBM, CatBoost) не смогли переносимо побить лучший боевой результат. **Практический потолок честного однопайплайнового подхода ≈0.388, и лучший паблик 0.38803 зафиксирован.** Разрыв до «стены» ~0.3735 — это растиражированный тюненый ансамбль, а не цель для одномодельной работы. Главная ценность проекта — методология: честная итеративная валидация без утечек, гейт 3 окон и задокументированная карта того, что переносится на лидерборд, а что нет.